# BioStudies — Biological Study Descriptions and Cross-Repository Links

**BioStudies** is EMBL-EBI's database for storing biological study descriptions and their associated data across multiple repositories. Each entry captures the full context of a study: experimental metadata, links to publications, and pointers to data files deposited in specialised EBI resources (ArrayExpress, BioImage Archive, PRIDE, EVA, etc.).

Key data types:
| Data type | Description |
|---|---|
| **Studies** | Experiment descriptions with accessions like `S-EPMC\d+`, `S-BIAD\d+`, `S-DIXA\d+` |
| **Sections** | Hierarchical metadata blocks (Author, Publication, Sample, Assay, …) |
| **Links** | Pointers to external databases (EuropePMC, ArrayExpress, Ensembl, …) |
| **Files** | Data files attached to a study or its subsections |
| **File trees** | Directory-level listing of all files for a study |

**Reference:** Sarkans et al. (2021), *Nucleic Acids Research*, BioStudies database — one stop shop for all information about a biological study

**API base:** `https://www.ebi.ac.uk/biostudies/api/v1`

# TODO

* [x] **Ingest data**
    * [x] Connect to BioStudies API and confirm access (fetch one study, print key fields)
    * [x] Search for studies with a broad term ("proteomics") and page through results
    * [x] Cache raw search results to `data/biostudies_studies.json`
    * [x] Parse into a Polars DataFrame: accession, title, study_type, organism, release_date, file_count, linked_databases
    * [x] Fetch full metadata for a study of interest and display its section/subsection structure
    * [x] Print DataFrame shape, dtypes, and head
* [ ] **Explore and clean**
    * [ ] Summarise study counts by type, organism, and year
    * [ ] Inspect linked_databases distribution — which EBI resources are most cross-referenced?
    * [ ] Handle missing/null metadata fields
* [ ] **Analysis**
    * [ ] Identify trends in proteomics submissions over time
    * [ ] Cluster studies by organism and linked database profile
    * [ ] Explore file_count distributions and identify unusually large studies
* [ ] **Visualization**
    * [ ] Bar charts of studies by organism and submission year
    * [ ] Heatmap of study type × linked database co-occurrence
    * [ ] Network diagram of studies linked to multiple EBI resources
* [ ] **Statistical analysis**
    * [ ] Discuss multiple hypothesis correction when testing enrichment across study metadata
    * [ ] Compare organism distributions across study types (chi-squared / Fisher's exact)

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

## 1. Ingest Data

### 1.1 Connect to BioStudies API and Confirm Access

In [ ]:
BIOSTUDIES_BASE = "https://www.ebi.ac.uk/biostudies/api/v1"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def bs_get(endpoint: str, params: dict = None) -> dict:
    """
    Send a GET request to the BioStudies REST API.

    Parameters
    ----------
    endpoint : str
        API path relative to BIOSTUDIES_BASE (e.g. "studies/S-EPMC6267371").
    params : dict, optional
        Query parameters to include in the request.

    Returns
    -------
    dict
        Parsed JSON response body.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.
    """
    url = f"{BIOSTUDIES_BASE}/{endpoint}"
    resp = requests.get(url, params=params or {}, timeout=30)
    resp.raise_for_status()
    return resp.json()


# Connectivity check: fetch a single known study
SAMPLE_ACCESSION = "S-EPMC6267371"
study = bs_get(f"studies/{SAMPLE_ACCESSION}")

# The top-level 'accno' field holds the accession; 'section' holds study metadata
accession   = study.get("accno")
title       = next(
    (a["value"] for a in study.get("attributes", []) if a.get("name") == "Title"),
    "N/A",
)
rel_date    = next(
    (a["value"] for a in study.get("attributes", []) if a.get("name") == "ReleaseDate"),
    "N/A",
)

print(f"Accession    : {accession}")
print(f"Title        : {title}")
print(f"Release date : {rel_date}")

### 1.2 Search for Studies and Page Through Results

In [ ]:
SEARCH_CACHE = DATA_DIR / "biostudies_studies.json"
SEARCH_QUERY = "proteomics"
PAGE_SIZE    = 100   # maximum page size accepted by the BioStudies search endpoint


def fetch_all_studies(
    query: str,
    cache_path: Path = SEARCH_CACHE,
    page_size: int = PAGE_SIZE,
) -> list[dict]:
    """
    Search BioStudies for a query term and page through all results.

    The search endpoint returns a 'hits' array inside a top-level JSON object.
    Pagination is controlled by the 'page' (1-indexed) and 'pageSize' parameters.
    Results are cached to disk; if the cache exists the download is skipped.

    Parameters
    ----------
    query : str
        Free-text search term (e.g. "proteomics").
    cache_path : Path
        File path for the JSON cache.
    page_size : int
        Records to request per page (max 100).

    Returns
    -------
    list[dict]
        One dict per study hit, exactly as returned by the API.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_hits: list[dict] = []
    page = 1   # BioStudies search uses 1-based page numbering

    while True:
        resp = bs_get(
            "search",
            {"query": query, "page": page, "pageSize": page_size},
        )
        hits = resp.get("hits", [])
        if not hits:
            break                          # no more results

        all_hits.extend(hits)
        total = resp.get("totalHits", "?")
        print(f"  Page {page:>4} — {len(all_hits):>6} / {total} studies", end="\r")

        # Stop early if we have collected all available hits
        if isinstance(total, int) and len(all_hits) >= total:
            break

        page += 1
        time.sleep(0.3)   # polite delay between paginated requests

    print(f"\nDone. Fetched {len(all_hits)} studies for query '{query}'.")
    cache_path.write_text(json.dumps(all_hits))   # persist to disk
    return all_hits


studies_raw = fetch_all_studies(SEARCH_QUERY)
print(f"Total cached studies: {len(studies_raw)}")

### 1.3 Parse Into a Polars DataFrame

In [ ]:
def extract_attribute(attributes: list[dict], name: str) -> str | None:
    """
    Extract the first matching attribute value from a BioStudies attribute list.

    BioStudies represents metadata as a list of ``{"name": ..., "value": ...}``
    dicts. This helper finds the first entry whose ``name`` matches and returns
    its ``value``.

    Parameters
    ----------
    attributes : list[dict]
        Attribute list from a study or section record.
    name : str
        Attribute name to look up (case-sensitive).

    Returns
    -------
    str or None
        The attribute value string, or ``None`` if not found.
    """
    return next(
        (a["value"] for a in (attributes or []) if a.get("name") == name),
        None,
    )


def flatten_study(hit: dict) -> dict:
    """
    Flatten a single BioStudies search-hit dict into a row-compatible format.

    Search hits already contain a small set of pre-computed fields; nested
    list fields (organisms, links, etc.) are joined as pipe-separated strings
    so each study occupies exactly one DataFrame row.

    Parameters
    ----------
    hit : dict
        Raw study hit record from the BioStudies search API.

    Returns
    -------
    dict
        Flat dict suitable for constructing a Polars DataFrame row.
    """
    # 'links' is a list of dicts with 'url' and 'type' keys pointing to
    # related resources such as ArrayExpress, EuropePMC, PRIDE, etc.
    links = hit.get("links", []) or []
    linked_dbs = " | ".join(
        lnk.get("type", "")
        for lnk in links
        if lnk.get("type")          # skip entries with no type label
    ) or None

    return {
        "accession":        hit.get("accession"),
        "title":            hit.get("title"),
        "study_type":       hit.get("type"),          # e.g. "compound", "study"
        "organism":         hit.get("organism"),       # pre-flattened by the search index
        "release_date":     hit.get("releaseDate"),
        "file_count":       hit.get("filesCount"),
        "linked_databases": linked_dbs,
    }


rows = [flatten_study(h) for h in studies_raw]

studies = (
    pl.DataFrame(rows)
    .with_columns([
        # Parse ISO-8601 date string; keep nulls for missing values
        pl.col("release_date").str.to_date("%Y-%m-%d", strict=False),
        pl.col("file_count").cast(pl.Int32, strict=False),
    ])
)

print(f"Shape  : {studies.shape}")
print(f"Memory : {studies.estimated_size('kb'):.1f} KB\n")
print(studies.dtypes)
studies.head(5)

### 1.4 Fetch Full Metadata for a Study of Interest

In [ ]:
FOCUS_ACCESSION = "S-EPMC6267371"   # a proteomics study cross-linked to multiple EBI resources


def describe_section(section: dict, depth: int = 0) -> None:
    """
    Recursively print the section/subsection hierarchy of a BioStudies study.

    BioStudies organises metadata into a tree of Sections, each with a type
    (e.g. "Study", "Author", "Publication", "Sample"), a list of attributes,
    and optionally nested subsections and links.

    Parameters
    ----------
    section : dict
        A section dict as returned by the ``/studies/{accession}`` endpoint.
    depth : int
        Current recursion depth; controls indentation for display.
    """
    indent  = "  " * depth
    sec_type = section.get("type", "—")
    accno    = section.get("accno", "")
    attrs    = section.get("attributes", []) or []
    n_links  = len(section.get("links", []) or [])
    n_files  = len(section.get("files", []) or [])

    # Build a compact summary line: type, accno (if present), attribute count, links, files
    summary_parts = [f"{indent}[{sec_type}]"]
    if accno:
        summary_parts.append(f"accno={accno}")
    summary_parts.append(f"{len(attrs)} attr(s)")
    if n_links:
        summary_parts.append(f"{n_links} link(s)")
    if n_files:
        summary_parts.append(f"{n_files} file(s)")
    print("  ".join(summary_parts))

    # Print each attribute name→value on its own indented line
    for attr in attrs:
        name  = attr.get("name", "?")
        value = attr.get("value", "")
        # Truncate long values to keep output readable
        display = value[:80] + "…" if len(str(value)) > 80 else value
        print(f"{indent}    {name}: {display}")

    # Recurse into subsections
    for sub in section.get("subsections", []) or []:
        describe_section(sub, depth + 1)


# Fetch the complete record for the focus study
detail = bs_get(f"studies/{FOCUS_ACCESSION}")

print(f"=== {FOCUS_ACCESSION} — top-level attributes ===")
for attr in detail.get("attributes", []):
    print(f"  {attr.get('name')}: {attr.get('value', '')[:100]}")

print(f"\n=== Section / subsection tree ===")
root_section = detail.get("section", {})
describe_section(root_section)

### 1.5 DataFrame Shape, Dtypes, and Head

In [ ]:
# ── Shape ────────────────────────────────────────────────────────────────────
print(f"Shape  : {studies.shape[0]} rows × {studies.shape[1]} columns")
print(f"Memory : {studies.estimated_size('kb'):.1f} KB\n")

# ── Dtypes ───────────────────────────────────────────────────────────────────
print("Column dtypes:")
for col, dtype in zip(studies.columns, studies.dtypes):
    print(f"  {col:<20} {dtype}")

# ── Null counts ──────────────────────────────────────────────────────────────
print("\nNull counts per column:")
print(studies.null_count())

# ── Head ─────────────────────────────────────────────────────────────────────
print("\nFirst 5 rows:")
studies.head(5)